In [ ]:
# === NEW: Download a small image dataset from Panoramas of Cinema (PoC) ===
# Target page (your request):
POC_SEARCH_URL = "https://panoramasofcinema.ch/search?q=places=arcade,0.010,1.000+faces=no_faces,small_faces"

# How many images to download (start small; increase later)
MAX_IMAGES = 120

# Where to save the downloaded images
from pathlib import Path
POC_OUT_DIR = Path("dataset/poc_arcade")
POC_OUT_DIR.mkdir(parents=True, exist_ok=True)

print("PoC output dir:", POC_OUT_DIR.resolve())

# Install small dependencies (safe in Jupyter)
import sys
!{sys.executable} -m pip -q install selenium requests

import time
import os
import hashlib
import requests
from urllib.parse import urljoin
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options

def build_driver(headless=True):
    opts = Options()
    if headless:
        opts.add_argument("--headless=new")
    opts.add_argument("--window-size=1400,900")
    opts.add_argument("--disable-gpu")
    opts.add_argument("--no-sandbox")
    opts.add_argument("--disable-dev-shm-usage")
    d = webdriver.Chrome(options=opts)
    d.implicitly_wait(10)
    return d

def extract_image_urls(driver, base_url):
    urls = set()

    # 1) <img> elements
    for img in driver.find_elements(By.TAG_NAME, "img"):
        for attr in ("src", "data-src", "data-lazy", "data-original"):
            try:
                u = img.get_attribute(attr)
            except Exception:
                u = None
            if u:
                urls.add(urljoin(base_url, u))

    # 2) background-image styles
    for el in driver.find_elements(By.CSS_SELECTOR, "*[style]"):
        try:
            st = el.get_attribute("style") or ""
        except Exception:
            st = ""
        if "background-image" in st and "url(" in st:
            m = re.findall(r'url\(["\']?(.*?)["\']?\)', st)
            for u in m:
                if u:
                    urls.add(urljoin(base_url, u))

    # 3) regex over page source (often catches urls created by JS)
    html = driver.page_source or ""
    for u in re.findall(r'https?://[^"\']+\.(?:jpg|jpeg|png|webp)(?:\?[^"\']*)?', html, flags=re.I):
        urls.add(u)

    # filter: keep likely image urls only
    def ok(u: str):
        u2 = u.lower()
        if any(x in u2 for x in ("sprite", "icon", "logo")):
            return False
        return any(u2.endswith(ext) or (ext in u2 and ".jpg" in u2 or ".png" in u2 or ".webp" in u2) for ext in (".jpg",".jpeg",".png",".webp"))

    return [u for u in urls if ok(u)]

def safe_name_from_url(url):
    h = hashlib.md5(url.encode("utf-8")).hexdigest()[:10]
    # guess extension
    m = re.search(r'\.(jpg|jpeg|png|webp)(?:\?|$)', url, flags=re.I)
    ext = (m.group(1).lower() if m else "jpg")
    if ext == "jpeg":
        ext = "jpg"
    return f"poc_{h}.{ext}"

def download(url, out_dir):
    fn = safe_name_from_url(url)
    path = out_dir / fn
    if path.exists() and path.stat().st_size > 1500:
        return str(path), ""
    try:
        r = requests.get(url, timeout=25, stream=True, headers={"User-Agent":"Mozilla/5.0"})
        r.raise_for_status()
        ctype = (r.headers.get("Content-Type") or "").lower()
        if "image" not in ctype:
            # still write if it looks like image by extension, but mark warning
            pass
        with open(path, "wb") as f:
            for chunk in r.iter_content(8192):
                if chunk:
                    f.write(chunk)
        if path.stat().st_size < 1500:
            return "", "file too small"
        return str(path), ""
    except Exception as e:
        return "", str(e)

# Only download if folder is still small (so rerunning notebook won't re-download)
existing = list(POC_OUT_DIR.glob("*"))
if len(existing) >= 20:
    print(f"[SKIP] Found {len(existing)} existing files in {POC_OUT_DIR}. Not downloading again.")
else:
    print("[INFO] Opening PoC page with Selenium...")
    driver = build_driver(headless=False)
    driver.get(POC_SEARCH_URL)
    time.sleep(2)

    collected = set()
    scroll_rounds = 25

    for i in range(scroll_rounds):
        # scroll to bottom to trigger lazy-load/infinite scroll (if any)
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(1.0)

        urls = extract_image_urls(driver, "https://panoramasofcinema.ch/")
        # keep new ones
        new_urls = [u for u in urls if u not in collected]
        if new_urls:
            print(f"[INFO] Scroll {i+1}/{scroll_rounds}: found {len(new_urls)} new image urls (total urls {len(collected)+len(new_urls)})")
        collected.update(new_urls)

        if len(collected) >= MAX_IMAGES:
            break

    driver.quit()

    print(f"[INFO] Total unique image urls collected: {len(collected)}")
    # download
    ok = 0
    fail = 0
    for u in list(collected)[:MAX_IMAGES]:
        path, err = download(u, POC_OUT_DIR)
        if path:
            ok += 1
        else:
            fail += 1
        if (ok + fail) % 20 == 0:
            print(f"[DL] {ok} ok / {fail} fail")

    print(f"[DONE] Downloaded {ok} images, failed {fail}. Folder: {POC_OUT_DIR.resolve()}")


In [1]:
from pathlib import Path

# 你 Workshop1 生成的 100 张图（默认路径：../Workshop1/workshop1_output/images/dataset_100）
W1_DATASET_DIR = POC_OUT_DIR  # downloaded PoC images

# Workshop2 输出目录
W2_OUT = Path("workshop2_output")
W2_OUT.mkdir(parents=True, exist_ok=True)

# 你自己的视频数据集（可选）
VIDEO_DIR = Path("dataset/videos")   # 把视频放这里
FRAMES_DIR = Path("dataset/frames")  # 抽帧输出这里
VIDEO_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

print("W1 dataset exists:", W1_DATASET_DIR.exists(), W1_DATASET_DIR.resolve())
print("W2 output:", W2_OUT.resolve())
print("Video dir:", VIDEO_DIR.resolve())
print("Frames dir:", FRAMES_DIR.resolve())

W1 dataset exists: False C:\Users\Workshop1\workshop1_output\images\dataset_100
W2 output: C:\Users\24586\workshop2_output
Video dir: C:\Users\24586\dataset\videos
Frames dir: C:\Users\24586\dataset\frames


In [2]:
from pathlib import Path

hits = []
for p in Path("..").rglob("dataset_100"):
    hits.append(p)

print("Found dataset_100 folders:")
for h in hits:
    print(" -", h.resolve())

Found dataset_100 folders:
 - C:\Users\24586\workshop1_output\images\dataset_100


In [3]:
import sys
!{sys.executable} -m pip -q install numpy pandas pillow tqdm opencv-python
!{sys.executable} -m pip -q install faiss-cpu open-clip-torch torch


[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: C:\Users\24586\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0
[notice] To update, run: C:\Users\24586\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [4]:
import cv2
from pathlib import Path
from tqdm import tqdm

VIDEO_EXTS = {".mp4", ".mov", ".mkv", ".avi", ".webm"}

def iter_videos(video_dir: Path):
    for p in video_dir.rglob("*"):
        if p.suffix.lower() in VIDEO_EXTS:
            yield p

def extract_frames(video_path: Path, out_dir: Path, fps: float = 1.0, max_frames: int = 0):
    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        return 0

    native_fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    step = max(int(round(native_fps / fps)), 1)

    out_dir.mkdir(parents=True, exist_ok=True)
    stem = video_path.stem

    saved = 0
    idx = 0
    frame_i = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break

        if idx % step == 0:
            out_path = out_dir / f"{stem}__frame_{frame_i:08d}.jpg"
            cv2.imwrite(str(out_path), frame)
            saved += 1
            frame_i += 1
            if max_frames and saved >= max_frames:
                break
        idx += 1

    cap.release()
    return saved

videos = list(iter_videos(VIDEO_DIR))
print("Found videos:", len(videos))

total = 0
for vp in tqdm(videos, desc="Extracting frames"):
    total += extract_frames(vp, FRAMES_DIR, fps=1.0, max_frames=0)

print("Saved frames:", total)

Found videos: 0


Extracting frames: 0it [00:00, ?it/s]

Saved frames: 0


In [5]:
from pathlib import Path

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

dataset_paths = []

# Workshop1 100 张图
if W1_DATASET_DIR.exists():
    dataset_paths += [p for p in W1_DATASET_DIR.glob("*") if p.suffix.lower() in IMG_EXTS]

# 视频帧（如果你抽了帧）
if FRAMES_DIR.exists():
    dataset_paths += [p for p in FRAMES_DIR.rglob("*") if p.suffix.lower() in IMG_EXTS]

dataset_paths = sorted(dataset_paths)
print("Total dataset images/frames:", len(dataset_paths))
dataset_paths[:5]

Total dataset images/frames: 0


[]

In [2]:
import os, glob

print("W1_DATASET_DIR =", W1_DATASET_DIR)

imgs = []
for ext in ("*.jpg","*.jpeg","*.png","*.webp","*.bmp"):
    imgs += glob.glob(os.path.join(W1_DATASET_DIR, ext))

print("Found images:", len(imgs))
print("First 5:", imgs[:5])


NameError: name 'W1_DATASET_DIR' is not defined

In [3]:
import numpy as np
import faiss
import torch
import open_clip
from PIL import Image
from tqdm import tqdm
import json

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "ViT-B-32"
pretrained = "openai"

model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=pretrained)
model = model.to(device)
model.eval()

@torch.inference_mode()
def encode_images(paths, batch_size=24):
    feats = []
    for i in tqdm(range(0, len(paths), batch_size), desc="CLIP encoding"):
        chunk = paths[i:i+batch_size]
        ims = [preprocess(Image.open(p).convert("RGB")) for p in chunk]
        inp = torch.stack(ims).to(device)
        emb = model.encode_image(inp)
        emb = emb / emb.norm(dim=-1, keepdim=True)
        feats.append(emb.detach().cpu().numpy().astype(np.float32))
    return np.vstack(feats)

embeddings = encode_images(dataset_paths, batch_size=24)
dim = embeddings.shape[1]

index = faiss.IndexFlatIP(dim)
index.add(embeddings)

faiss_path = W2_OUT / "clip_index.faiss"
meta_path = W2_OUT / "clip_meta_paths.json"
faiss.write_index(index, str(faiss_path))
with open(meta_path, "w", encoding="utf-8") as f:
    json.dump([str(p) for p in dataset_paths], f, ensure_ascii=False, indent=2)

print("Saved:", faiss_path)
print("Saved:", meta_path)
embeddings.shape

ModuleNotFoundError: No module named 'numpy'

In [4]:
import pandas as pd

@torch.inference_mode()
def encode_text(text: str):
    tok = open_clip.tokenize([text]).to(device)
    emb = model.encode_text(tok)
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.detach().cpu().numpy().astype(np.float32)

def search_by_text(prompt: str, top_k=30):
    q = encode_text(prompt)
    scores, ids = index.search(q, top_k)
    ids = ids[0].tolist()
    scores = scores[0].tolist()
    return [(dataset_paths[i], scores[j]) for j,i in enumerate(ids) if i >= 0]

PROMPTS = [
    "spatial illusion forced perspective trompe l'oeil mirror corridor impossible space",
    "optical illusion pattern moire op art high contrast",
    "time illusion chronostasis temporal perception",
    "sound illusion shepard tone auditory illusion"
]

all_rows = []
for prompt in PROMPTS:
    results = search_by_text(prompt, top_k=25)
    for rank, (p, sc) in enumerate(results, start=1):
        all_rows.append({"prompt": prompt, "rank": rank, "score": float(sc), "path": str(p)})

df_sim = pd.DataFrame(all_rows)
out_csv = W2_OUT / "similarity_text_results.csv"
df_sim.to_csv(out_csv, index=False, encoding="utf-8")
print("Saved:", out_csv)
df_sim.head(10)

ModuleNotFoundError: No module named 'pandas'

In [5]:
@torch.inference_mode()
def encode_one_image(path: str):
    im = preprocess(Image.open(path).convert("RGB")).unsqueeze(0).to(device)
    emb = model.encode_image(im)
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.detach().cpu().numpy().astype(np.float32)

def search_by_image(query_path: str, top_k=30):
    q = encode_one_image(query_path)
    scores, ids = index.search(q, top_k)
    ids = ids[0].tolist()
    scores = scores[0].tolist()
    return [(dataset_paths[i], scores[j]) for j,i in enumerate(ids) if i >= 0]

query_path = str(dataset_paths[0])  # 你也可以换成自己的图片路径
img_results = search_by_image(query_path, top_k=30)

df_img = pd.DataFrame([{"rank": i+1, "score": float(sc), "path": str(p)} for i,(p,sc) in enumerate(img_results)])
out_csv2 = W2_OUT / "similarity_image_results.csv"
df_img.to_csv(out_csv2, index=False, encoding="utf-8")
print("Saved:", out_csv2)
df_img.head(10)

NameError: name 'torch' is not defined

In [6]:
from PIL import Image, ImageDraw, ImageFont

def make_contact_sheet(items, out_path, cols=6, thumb=220):
    rows = (len(items) + cols - 1) // cols
    sheet = Image.new("RGB", (cols*thumb, rows*thumb), (255,255,255))
    draw = ImageDraw.Draw(sheet)
    try:
        font = ImageFont.truetype("DejaVuSans.ttf", 14)
    except Exception:
        font = ImageFont.load_default()

    for idx, (img_path, score) in enumerate(items):
        r = idx // cols
        c = idx % cols
        x, y = c*thumb, r*thumb
        try:
            im = Image.open(img_path).convert("RGB")
            im.thumbnail((thumb, thumb))
            sheet.paste(im, (x,y))
            draw.rectangle([x, y, x+70, y+22], fill=(255,255,255))
            draw.text((x+4, y+3), f"{score:.2f}", fill=(0,0,0), font=font)
        except Exception:
            continue

    out_path.parent.mkdir(parents=True, exist_ok=True)
    sheet.save(out_path)
    return out_path

# 用“空间错觉 prompt”那组前 30 张做 contact sheet
spatial_results = search_by_text(PROMPTS[0], top_k=30)
sheet1 = make_contact_sheet(spatial_results, W2_OUT / "contact_sheet_spatial_prompt.jpg")
print("Saved:", sheet1)

# 用 image query 前 30 张做 contact sheet
sheet2 = make_contact_sheet(img_results[:30], W2_OUT / "contact_sheet_image_query.jpg")
print("Saved:", sheet2)

ModuleNotFoundError: No module named 'PIL'

In [7]:
import sys
!{sys.executable} -m pip -q install ultralytics


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: C:\Users\16598\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [8]:
from ultralytics import YOLO
from PIL import Image
import pandas as pd
from tqdm import tqdm
from pathlib import Path

YOLO_OUT = W2_OUT / "yolo_out"
ANN_DIR = YOLO_OUT / "annotated"
CROP_DIR = YOLO_OUT / "crops"
ANN_DIR.mkdir(parents=True, exist_ok=True)
CROP_DIR.mkdir(parents=True, exist_ok=True)

yolo = YOLO("yolov8n-seg.pt")  # 第一次会下载权重

run_paths = dataset_paths[:20]  # 改成 50 或 100 都可以（越大越慢）
rows = []

for p in tqdm(run_paths, desc="YOLO segment"):
    res = yolo.predict(source=str(p), conf=0.35, save=False, verbose=False)[0]

    # 保存标注图
    ann = res.plot()  # BGR numpy
    ann_img = Image.fromarray(ann[..., ::-1])
    ann_path = ANN_DIR / Path(p).name
    ann_img.save(ann_path)

    if res.boxes is None or len(res.boxes) == 0:
        continue

    orig = Image.open(p).convert("RGB")
    W, H = orig.size

    for j, box in enumerate(res.boxes):
        cls_id = int(box.cls.item())
        cls_name = res.names.get(cls_id, str(cls_id))
        conf = float(box.conf.item())
        x1, y1, x2, y2 = box.xyxy.cpu().numpy().reshape(-1).tolist()
        x1, y1, x2, y2 = map(int, [max(0,x1), max(0,y1), min(W,x2), min(H,y2)])

        crop = orig.crop((x1,y1,x2,y2))
        crop_path = CROP_DIR / f"{Path(p).stem}__{cls_name}__{j:03d}.png"
        crop.save(crop_path)

        rows.append({
            "image_path": str(p),
            "annotated_path": str(ann_path),
            "crop_path": str(crop_path),
            "class": cls_name,
            "conf": conf,
            "x1": x1, "y1": y1, "x2": x2, "y2": y2
        })

df_det = pd.DataFrame(rows)
det_csv = YOLO_OUT / "detections.csv"
df_det.to_csv(det_csv, index=False, encoding="utf-8")

print("Saved:", det_csv)
df_det.head(10)

Creating new Ultralytics Settings v0.0.6 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\16598\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


ModuleNotFoundError: No module named 'pandas'

In [ ]:
import sys
# 如果没装过 SAM，先装（可能需要点时间）
!{sys.executable} -m pip -q install segment-anything

In [ ]:
import numpy as np
import cv2
import torch
from PIL import Image
import pandas as pd
from tqdm import tqdm
from pathlib import Path

from segment_anything import sam_model_registry, SamAutomaticMaskGenerator

SAM_OUT = W2_OUT / "sam_out"
OVERLAY_DIR = SAM_OUT / "overlays"
MASK_DIR = SAM_OUT / "masks_png"
CUTOUT_DIR = SAM_OUT / "cutouts"
for d in [OVERLAY_DIR, MASK_DIR, CUTOUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ✅ 改成你的 checkpoint 路径，例如：
# SAM_CHECKPOINT = r"C:\Users\you\Downloads\sam_vit_h_4b8939.pth"
SAM_CHECKPOINT = ""
SAM_MODEL_TYPE = "vit_h"   # vit_h / vit_l / vit_b
MAX_SAM_IMAGES = 8         # SAM 很慢，建议先 5~10 张

if not SAM_CHECKPOINT or not Path(SAM_CHECKPOINT).exists():
    raise SystemExit("SAM_CHECKPOINT 没设置或文件不存在：请下载 .pth 并填入本地路径后再运行。")

sam = sam_model_registry[SAM_MODEL_TYPE](checkpoint=SAM_CHECKPOINT)
sam.to(device=device)

generator = SamAutomaticMaskGenerator(
    model=sam,
    points_per_side=32,
    pred_iou_thresh=0.88,
    stability_score_thresh=0.92,
    crop_n_layers=1,
    crop_n_points_downscale_factor=2,
    min_mask_region_area=200,
)

def overlay_mask(rgb, mask):
    overlay = rgb.copy()
    color = np.array([0,255,0], dtype=np.uint8)
    overlay[mask] = (0.5*overlay[mask] + 0.5*color).astype(np.uint8)
    return overlay

def mask_to_cutout(rgb, mask):
    rgba = np.zeros((rgb.shape[0], rgb.shape[1], 4), dtype=np.uint8)
    rgba[..., :3] = rgb
    rgba[..., 3] = (mask.astype(np.uint8) * 255)
    return Image.fromarray(rgba)

rows = []
run_paths = dataset_paths[:MAX_SAM_IMAGES]

for p in tqdm(run_paths, desc="SAM masks"):
    bgr = cv2.imread(str(p))
    if bgr is None:
        continue
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    masks = generator.generate(rgb)

    preview = rgb.copy()
    for m in masks[:25]:
        preview = overlay_mask(preview, m["segmentation"])
    Image.fromarray(preview).save(OVERLAY_DIR / Path(p).name)

    for i, m in enumerate(masks):
        seg = m["segmentation"]
        mask_img = (seg.astype(np.uint8) * 255)
        mask_path = MASK_DIR / f"{Path(p).stem}__mask_{i:04d}.png"
        Image.fromarray(mask_img).save(mask_path)

        cutout = mask_to_cutout(rgb, seg)
        cutout_path = CUTOUT_DIR / f"{Path(p).stem}__cutout_{i:04d}.png"
        cutout.save(cutout_path)

        rows.append({
            "image_path": str(p),
            "mask_path": str(mask_path),
            "cutout_path": str(cutout_path),
            "area": int(m.get("area", 0)),
            "pred_iou": float(m.get("pred_iou", 0.0)),
            "stability_score": float(m.get("stability_score", 0.0)),
        })

df_sam = pd.DataFrame(rows)
sam_csv = SAM_OUT / "masks.csv"
df_sam.to_csv(sam_csv, index=False, encoding="utf-8")

print("Saved:", sam_csv)
df_sam.head(10)